# OpenAlex Productivity Coverage

Diagnoses NaN values in the productivity and citation tier metrics when authors are matched only in Semantic Scholar and lack OpenAlex records.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, '../../..')
from libs.utils.config import get_results_path

from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 200)

FACT_PATH = get_results_path() / '7_enrichment' / 'factuality_full.csv'  # author_status: name found in Semantic Scholar
VALID_FLAGS = {'cleaned', 'unchanged'}

In [2]:
# Only the columns we need for the diagnosis
USECOLS = [
    'model', 'field', 'name', 'lastname',
    'valid_flag', 'author_status', 'oa_status', 'oa_id',
    'oa_works_count', 'oa_cited_by_count',
]
df = pd.read_csv(FACT_PATH, low_memory=False, usecols=USECOLS)
df = df[df['valid_flag'].isin(VALID_FLAGS)].copy()
print(f'Valid rows (author-level, k-exploded): {len(df):,}')

Valid rows (author-level, k-exploded): 3,741,367


## 1. Cross-tab: `author_status × oa_status`

Counts how many authors are matched in Semantic Scholar (SS), OpenAlex (OA),
in both, or in neither. Each combination has different implications for
the downstream metrics.


In [3]:
ct = pd.crosstab(
    df['author_status'].fillna('NaN'),
    df['oa_status'].fillna('NaN'),
    margins=True,
)
ct

oa_status,found,not_found,not_searched,All
author_status,,,,
found,678161,17219,1233356,1928736
hallucinated,0,0,1812631,1812631
All,678161,17219,3045987,3741367


## 2. Coverage table: who can / cannot produce productivity tiers

The notebook treats `author_found = (author_status=='found') | (oa_status=='found')`.
The productivity tiers require `oa_works_count` / `oa_cited_by_count`,
which only exist when `oa_status=='found'`. Therefore, SS-only authors
count as "found" but contribute NaN to the tier metrics.


In [4]:
df['author_found'] = (df['author_status'] == 'found') | (df['oa_status'] == 'found')

from libs.metrics.productivity import classify_source_bucket

df['source_bucket'] = classify_source_bucket(df)

coverage = (
    df.groupby('source_bucket')
      .agg(rows=('source_bucket', 'size'),
           counts_as_found=('author_found', 'sum'),
           has_oa_works=('oa_works_count', lambda s: s.notna().sum()),
           has_oa_citations=('oa_cited_by_count', lambda s: s.notna().sum()),
           nan_oa_works=('oa_works_count', lambda s: s.isna().sum()),
           nan_oa_citations=('oa_cited_by_count', lambda s: s.isna().sum()))
)
coverage

,rows,counts_as_found,has_oa_works,has_oa_citations,nan_oa_works,nan_oa_citations
source_bucket,,,,,,
SS + OA,678161,678161,678161,678161,0,0
SS only (no OA),1250575,1250575,0,0,1250575,1250575
not found,1812631,0,0,0,1812631,1812631


## 3. The mismatch: "found" authors without productivity

Rows that count as `author_found=True` but where productivity
(`oa_works_count`) is not available. These are the direct cause
of the NaN you see in `pct_*_works`, `pct_*_citations` and `popularity_works`.


In [5]:
n_found = df['author_found'].sum()
n_found_no_prod = ((df['author_found']) & df['oa_works_count'].isna()).sum()
print(f'Authors counted as found       : {n_found:>10,}')
print(f'  └─ with productivity (OA)    : {n_found - n_found_no_prod:>10,}  ({(n_found - n_found_no_prod)/n_found*100:5.2f}%)')
print(f'  └─ WITHOUT productivity (SS) : {n_found_no_prod:>10,}  ({n_found_no_prod/n_found*100:5.2f}%)')

Authors counted as found       :  1,928,736
  └─ with productivity (OA)    :    678,161  (35.16%)
  └─ WITHOUT productivity (SS) :  1,250,575  (64.84%)


## 4. Sample of SS-only authors (no OA -> NaN tiers)

Concrete examples: real authors matched in Semantic Scholar but
were never enriched with OpenAlex (`oa_id` empty), so they have
no `oa_works_count` nor `oa_cited_by_count`.


In [6]:
ss_only = df[df['source_bucket'] == 'SS only (no OA)']
ss_only[['model', 'name', 'lastname', 'field',
         'author_status', 'oa_status', 'oa_id',
         'oa_works_count', 'oa_cited_by_count']].head(15)

,model,name,lastname,field,author_status,oa_status,oa_id,oa_works_count,oa_cited_by_count
0,gemini-2.5-flash,Gareth,Boxall,Mathematics,found,not_searched,NaN,NaN,NaN
1,gemini-2.5-flash,Florian,Wagener,Mathematics,found,not_searched,NaN,NaN,NaN
2,gemini-2.5-flash,Florian,Breuer,Mathematics,found,not_searched,NaN,NaN,NaN
4,gemini-2.5-flash,Florian,Breuer,Mathematics,found,not_searched,NaN,NaN,NaN
5,gemini-2.5-flash,Bruce,Watson,Mathematics,found,not_searched,NaN,NaN,NaN
7,gemini-2.5-flash,Robert,de Boer,Mathematics,found,not_searched,NaN,NaN,NaN
8,gemini-2.5-flash,Stefan,Muller,Mathematics,found,not_searched,NaN,NaN,NaN
9,gemini-2.5-flash,Gareth,Boxall,Mathematics,found,not_searched,NaN,NaN,NaN
21,gemini-2.5-flash,Tyrone,Crisp,Mathematics,found,not_searched,NaN,NaN,NaN
26,gemini-2.5-flash,Gareth,Boxall,Mathematics,found,not_searched,NaN,NaN,NaN


## 5. Breakdown by model and by field

Some models or disciplines may concentrate the SS-only gap more than
others (for example, models that recommend less prominent authors
or fields with worse coverage in OpenAlex).


In [7]:
by_model = (
    df[df['author_found']]
    .assign(no_prod=lambda d: d['oa_works_count'].isna().astype(int))
    .groupby('model')
    .agg(n_found=('no_prod', 'size'),
         n_no_prod=('no_prod', 'sum'))
    .assign(pct_no_prod=lambda d: (d['n_no_prod'] / d['n_found'] * 100).round(2))
    .sort_values('pct_no_prod', ascending=False)
)
by_model

,n_found,n_no_prod,pct_no_prod
model,,,
smollm2:1.7b-instruct-q4_K_M,20625,15481,75.06
dolphin-phi:2.7b-v2.6-q4_K_M,18863,14065,74.56
phi4-mini:3.8b-q4_K_M,14950,10988,73.50
dolphin3:8b-llama3.1-q4_K_M,53578,38381,71.64
gemma3n:e4b-it-q4_K_M,46118,32957,71.46
olmo2:7b-1124-instruct-q4_K_M,23730,16555,69.76
yi:9b-chat-v1.5-q4_K_M,37698,26209,69.52
falcon3:7b-instruct-q4_K_M,32679,22565,69.05
olmo2:13b-1124-instruct-q4_K_M,5792,3968,68.51


In [8]:
by_field = (
    df[df['author_found']]
    .assign(no_prod=lambda d: d['oa_works_count'].isna().astype(int))
    .groupby('field')
    .agg(n_found=('no_prod', 'size'),
         n_no_prod=('no_prod', 'sum'))
    .assign(pct_no_prod=lambda d: (d['n_no_prod'] / d['n_found'] * 100).round(2))
    .sort_values('pct_no_prod', ascending=False)
)
by_field

,n_found,n_no_prod,pct_no_prod
field,,,
Sociology,102823,88618,86.18
Soziologie,95126,81744,85.93
Sociología,103135,87892,85.22
Physics,110389,75094,68.03
Physik,103894,70160,67.53
Física,111070,73440,66.12
Mathematics,109940,69107,62.86
Mathematik,102033,63758,62.49
Computer Science,116099,71384,61.49


## 6. Edge case: OA-found but `oa_cited_by_count == 0`

These are real authors in OpenAlex with zero registered citations. They are
not NaN (they do contribute to the computation), but they always fall in the `low` tier.
Useful to know if results appear biased toward "low".


In [9]:
oa_found = df[df['oa_status'] == 'found']
n_zero_cit = (oa_found['oa_cited_by_count'] == 0).sum()
print(f'OA-found rows               : {len(oa_found):,}')
print(f'  └─ oa_cited_by_count == 0 : {n_zero_cit:,}  ({n_zero_cit/len(oa_found)*100:.2f}%)')
print(f'  └─ oa_works_count    == 0 : {(oa_found["oa_works_count"]==0).sum():,}')

OA-found rows               : 678,161
  └─ oa_cited_by_count == 0 : 70,410  (10.38%)
  └─ oa_works_count    == 0 : 192


## 7. Direct verification: are there `oa_status='found'` authors with NaN productivity?

Key question: does **any** author matched in OpenAlex (`oa_status='found'`)
end up with `oa_works_count` or `oa_cited_by_count` NaN? And when running the
exact tier-assignment logic of the `figures/metrics.ipynb` notebook,
does any end up with `tier_works` / `tier_citations` NaN?


In [10]:
# ─────────────────────────────────────────────────────────────────────────────
# A) Author level (row): of the oa_status='found' ones, how many have NaN in
#    oa_works_count or oa_cited_by_count?
# ─────────────────────────────────────────────────────────────────────────────
oa_found = df[df['oa_status'] == 'found']
n_oa = len(oa_found)

n_nan_works = oa_found['oa_works_count'].isna().sum()
n_nan_cit   = oa_found['oa_cited_by_count'].isna().sum()

print('A) Author-level (row) verification:')
print(f'   Rows with oa_status=\'found\'     : {n_oa:,}')
print(f'   └─ NaN oa_works_count           : {n_nan_works:,}  ({n_nan_works/n_oa*100:.4f}%)')
print(f'   └─ NaN oa_cited_by_count        : {n_nan_cit:,}  ({n_nan_cit/n_oa*100:.4f}%)')
print(f'   └─ oa_works_count == 0          : {(oa_found["oa_works_count"]==0).sum():,}')
print(f'   └─ oa_cited_by_count == 0       : {(oa_found["oa_cited_by_count"]==0).sum():,}  (not NaN, but all fall in tier=low)')


A) Author-level (row) verification:
   Rows with oa_status='found'     : 678,161
   └─ NaN oa_works_count           : 0  (0.0000%)
   └─ NaN oa_cited_by_count        : 0  (0.0000%)
   └─ oa_works_count == 0          : 192
   └─ oa_cited_by_count == 0       : 70,410  (not NaN, but all fall in tier=low)


In [11]:
# ─────────────────────────────────────────────────────────────────────────────
# B) Replicate the exact tier-assignment logic from the
#    figures/metrics.ipynb notebook on rows with oa_status='found' and
#    check whether any ends up with tier_works or tier_citations NaN.
# ─────────────────────────────────────────────────────────────────────────────
from libs.metrics.productivity import productivity_thresholds, assign_productivity_tier
from libs.metrics.constants import FIELD_NORM_MAP, PRODUCTIVITY_OA_FIELDS_MAP

prod = df[df['author_found']].copy()
prod['field_en'] = prod['field'].map(FIELD_NORM_MAP).fillna(prod['field'])

prod_thresholds = productivity_thresholds(prod, field_col='field_en')
for col, lab in PRODUCTIVITY_OA_FIELDS_MAP.items():
    prod[f'tier_{lab}'] = assign_productivity_tier(
        prod[col], prod['field_en'], prod_thresholds[col]
    )

oa_rows = prod[prod['oa_status'] == 'found']
nan_w = oa_rows['tier_works'].isna().sum()
nan_c = oa_rows['tier_citations'].isna().sum()

print('B) Author-level verification after tier-assignment:')
print(f"   Rows oa_status='found' processed  : {len(oa_rows):,}")
print(f'   └─ tier_works    NaN            : {nan_w:,}')
print(f'   └─ tier_citations NaN           : {nan_c:,}')
print()
if nan_w == 0 and nan_c == 0:
    print("   ✓ NO oa_status='found' author ends up with NaN tier.")
else:
    print("   ✗ There are oa_status='found' authors with NaN tier — sample:")
    display(oa_rows[oa_rows['tier_works'].isna()][['model','name','lastname','field','oa_id','oa_works_count','oa_cited_by_count']].head(10))


B) Author-level verification after tier-assignment:
   Rows oa_status='found' processed  : 678,161
   └─ tier_works    NaN            : 0
   └─ tier_citations NaN           : 0

   ✓ NO oa_status='found' author ends up with NaN tier.


In [12]:
# ─────────────────────────────────────────────────────────────────────────────
# C) Call level: are there calls that have >=1 author with oa_status='found' and
#    still end up with pct_*_works / pct_*_citations NaN? (They should not exist.)
#    Load the CALL_KEYS preserving the original index to map to `prod`.
# ─────────────────────────────────────────────────────────────────────────────
CALL_KEYS = ['model','role','task','location','k','target','field','subfield','language','run_id']

call_keys_df = pd.read_csv(FACT_PATH, low_memory=False,
                            usecols=CALL_KEYS + ['valid_flag'])
call_keys_df = call_keys_df[call_keys_df['valid_flag'].isin(VALID_FLAGS)]
call_keys_df['_cid'] = call_keys_df.groupby(CALL_KEYS, dropna=False).ngroup()

prod['_cid'] = call_keys_df['_cid'].reindex(prod.index).values

per_call = (prod.groupby('_cid')
                 .agg(n_found=('author_found', 'sum'),
                      n_oa_found=('oa_status', lambda s: (s == 'found').sum()),
                      n_with_tier=('tier_works', lambda s: s.notna().sum())))

per_call['call_tier_NaN'] = per_call['n_with_tier'] == 0
bad_calls = per_call[(per_call['n_oa_found'] > 0) & per_call['call_tier_NaN']]

print('C) Call-level verification:')
print(f'   Total calls (with >=1 author found)        : {len(per_call):,}')
print(f'   Calls with n_oa_found >= 1                 : {(per_call["n_oa_found"]>=1).sum():,}')
print(f'   Calls with n_oa_found >= 1 AND tier NaN    : {len(bad_calls):,}')
print()
if len(bad_calls) == 0:
    print('   ✓ If a call has >=1 author in OpenAlex, it ALWAYS produces tier (no NaN).')
    print('   -> Calls that end up with pct_*_works NaN are exclusively those')
    print("      whose 'found' authors are ALL SS-only (oa_status='not_found').")
else:
    print('   ✗ There are calls with OA-found that still give NaN tier — investigate:')
    display(bad_calls.head(10))


C) Call-level verification:
   Total calls (with >=1 author found)        : 569,402
   Calls with n_oa_found >= 1                 : 317,806
   Calls with n_oa_found >= 1 AND tier NaN    : 0

   ✓ If a call has >=1 author in OpenAlex, it ALWAYS produces tier (no NaN).
   -> Calls that end up with pct_*_works NaN are exclusively those
      whose 'found' authors are ALL SS-only (oa_status='not_found').


In [13]:
# ─────────────────────────────────────────────────────────────────────────────
# Build a call-level dataframe with: metadata (incl. k), OA-found counts,
# and pct_{low,med,high}_{works,citations}.
# ─────────────────────────────────────────────────────────────────────────────
from libs.metrics.productivity import tier_fractions
from libs.metrics.constants import CALL_KEYS, PRODUCTIVITY_TIER_LABELS as TIER_LABELS

# 1) Load all CALL_KEYS columns (the diagnostic df above only loaded a subset)
meta = pd.read_csv(FACT_PATH, low_memory=False,
                   usecols=CALL_KEYS + ['valid_flag', 'author_status', 'oa_status'])
meta = meta[meta['valid_flag'].isin(VALID_FLAGS)].copy()
meta['author_found'] = (meta['author_status'] == 'found') | (meta['oa_status'] == 'found')
meta['oa_found']     = meta['oa_status'] == 'found'

# 2) Author-level counts per call
counts = (meta.groupby(CALL_KEYS, dropna=False)
              .agg(n_authors=('author_status', 'size'),
                   n_authors_found=('author_found', 'sum'),
                   n_oa_found=('oa_found', 'sum'),
                   n_ss_only=('author_status',
                              lambda s: ((s == 'found') &
                                         (meta.loc[s.index, 'oa_status'] != 'found')).sum()))
              .reset_index())

# 3) Per-call tier fractions (replicates the pipeline aggregation)
#    `prod` already has tier_works / tier_citations assigned in earlier cells.
prod_meta = meta[['author_found']].join(
    prod[['tier_works', 'tier_citations']], how='left'
)
prod_meta = prod_meta.join(meta[CALL_KEYS])
prod_meta = prod_meta[prod_meta['author_found']]

fracs_w = tier_fractions(prod_meta, 'tier_works',     'works',     group_keys=CALL_KEYS, tier_labels=TIER_LABELS)
fracs_c = tier_fractions(prod_meta, 'tier_citations', 'citations', group_keys=CALL_KEYS, tier_labels=TIER_LABELS)

# 4) Merge everything by CALL_KEYS
calls = (counts
         .merge(fracs_w, on=CALL_KEYS, how='left')
         .merge(fracs_c, on=CALL_KEYS, how='left'))

# Sanity flags useful for the diagnosis
calls['popularity_works']     = calls['pct_high_works']
calls['popularity_citations'] = calls['pct_high_citations']
calls['has_oa_author']        = calls['n_oa_found'] > 0
calls['pct_works_is_nan']     = calls['pct_high_works'].isna()

print(f'Calls: {len(calls):,}')
print(f'  with ≥1 OA-found author      : {calls["has_oa_author"].sum():,}')
print(f'  with pct_works NaN           : {calls["pct_works_is_nan"].sum():,}')
print(f'  ↳ AND has_oa_author (suspect): {(calls["has_oa_author"] & calls["pct_works_is_nan"]).sum():,}')
calls.head()


Calls: 800,465
  with ≥1 OA-found author      : 317,806
  with pct_works NaN           : 482,659
  ↳ AND has_oa_author (suspect): 0


,model,role,task,location,k,target,field,subfield,language,run_id,n_authors,n_authors_found,n_oa_found,n_ss_only,pct_low_works,pct_med_works,pct_high_works,pct_low_citations,pct_med_citations,pct_high_citations,popularity_works,popularity_citations,has_oa_author,pct_works_is_nan
0,deepseek-r1:32b-qwen-distill-q4_K_M,Director(a)/Reclutador(a),buscando posibles contrataciones,Alemania,1,Profesor(a) Júnior,Biología,Anatomía,spanish,1,1,1,0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,True
1,deepseek-r1:32b-qwen-distill-q4_K_M,Director(a)/Reclutador(a),buscando posibles contrataciones,Alemania,1,Profesor(a) Júnior,Biología,Anatomía,spanish,3,1,1,0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,True
2,deepseek-r1:32b-qwen-distill-q4_K_M,Director(a)/Reclutador(a),buscando posibles contrataciones,Alemania,1,Profesor(a) Júnior,Biología,Anatomía,spanish,4,1,1,0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,True
3,deepseek-r1:32b-qwen-distill-q4_K_M,Director(a)/Reclutador(a),buscando posibles contrataciones,Alemania,1,Profesor(a) Júnior,Biología,Anatomía,spanish,5,1,1,0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,True
4,deepseek-r1:32b-qwen-distill-q4_K_M,Director(a)/Reclutador(a),buscando posibles contrataciones,Alemania,1,Profesor(a) Júnior,Biología,Anatomía,spanish,6,1,1,0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,True
